In [1]:
import pandas as pd
import mysql.connector

print("Pandas:", pd.__version__)
print("MySQL Connector: Working")

Pandas: 3.0.5
MySQL Connector: Working


In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.db_connection import get_connection

connection = get_connection()

print("DATABASE CONNECTION SUCCESSFUL")

DATABASE CONNECTION SUCCESSFUL


In [3]:
cursor = connection.cursor()

cursor.execute("SHOW TABLES")

tables = [table[0] for table in cursor.fetchall()]

print("Tables in database:")
for table in tables:
    print("-", table)

Tables in database:
- downtime
- employees
- inventory
- machines
- maintenance
- production
- production_logs
- production_targets
- quality
- sensors
- shifts
- suppliers


In [4]:
data = {}

for table in tables:
    query = f"SELECT * FROM `{table}`"
    data[table] = pd.read_sql(query, connection)

    print(f"{table}: {data[table].shape[0]} rows × {data[table].shape[1]} columns")

downtime: 32 rows × 6 columns
employees: 5 rows × 5 columns
inventory: 5 rows × 6 columns
machines: 5 rows × 6 columns
maintenance: 13 rows × 7 columns
production: 450 rows × 7 columns
production_logs: 450 rows × 6 columns
production_targets: 450 rows × 4 columns
quality: 450 rows × 6 columns
sensors: 900 rows × 6 columns
shifts: 3 rows × 5 columns
suppliers: 5 rows × 6 columns


C:\Users\Sai Sanjana S\AppData\Local\Temp\ipykernel_31904\4262546910.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data[table] = pd.read_sql(query, connection)


In [5]:
for table, df in data.items():
    print("\n" + "=" * 60)
    print(f"TABLE: {table}")
    print("=" * 60)
    print("Columns:", list(df.columns))
    print("\nData types:")
    print(df.dtypes)


TABLE: downtime
Columns: ['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']

Data types:
downtime_id                 int64
machine_id                  int64
downtime_start     datetime64[us]
downtime_end       datetime64[us]
downtime_reason               str
downtime_hours            float64
dtype: object

TABLE: employees
Columns: ['employee_id', 'employee_name', 'department', 'role', 'shift']

Data types:
employee_id      int64
employee_name      str
department         str
role               str
shift              str
dtype: object

TABLE: inventory
Columns: ['inventory_id', 'material_name', 'quantity_available', 'reorder_level', 'unit', 'last_updated']

Data types:
inventory_id           int64
material_name            str
quantity_available     int64
reorder_level          int64
unit                     str
last_updated          object
dtype: object

TABLE: machines
Columns: ['machine_id', 'machine_name', 'machine_type', 'location',

In [6]:
summary = []

for table, df in data.items():
    summary.append({
        "table": table,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isnull().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)

summary_df

,table,rows,columns,missing_values,duplicate_rows
0,downtime,32,6,0,0
1,employees,5,5,0,0
2,inventory,5,6,0,0
3,machines,5,6,0,0
4,maintenance,13,7,0,0
5,production,450,7,0,0
6,production_logs,450,6,0,0
7,production_targets,450,4,0,0
8,quality,450,6,0,0
9,sensors,900,6,0,0


In [7]:
important_tables = [
    "machines",
    "production",
    "production_targets",
    "quality",
    "maintenance",
    "downtime",
    "sensors"
]

for table in important_tables:
    print("\n" + "=" * 70)
    print(f"TABLE: {table}")
    print("=" * 70)
    display(data[table])


TABLE: machines


,machine_id,machine_name,machine_type,location,status,installation_date
0,1,PCB Assembly Line 01,Assembly Line,Production Floor A,Operational,2022-05-10
1,2,PCB Assembly Line 02,Assembly Line,Production Floor A,Operational,2022-08-15
2,3,CNC Precision Unit,CNC Machine,Production Floor B,Maintenance,2021-03-20
3,4,Testing Station 01,Testing Equipment,Quality Floor,Operational,2023-01-12
4,5,Testing Station 02,Testing Equipment,Quality Floor,Operational,2023-04-18



TABLE: production


,production_id,machine_id,production_date,shift,units_produced,units_rejected,production_time_hours
0,1,1,2026-07-20,Morning,969,59,8.0
1,2,1,2026-07-20,Afternoon,944,48,8.0
2,3,1,2026-07-20,Night,940,13,8.0
3,4,2,2026-07-20,Morning,890,62,8.0
4,5,2,2026-07-20,Afternoon,963,11,8.0
...,...,...,...,...,...,...,...
445,446,4,2026-08-18,Afternoon,1050,42,8.0
446,447,4,2026-08-18,Night,1008,66,8.0
447,448,5,2026-08-18,Morning,907,53,8.0
448,449,5,2026-08-18,Afternoon,954,33,8.0



TABLE: production_targets


...

,maintenance_id,equipment_id,maintenance_date,maintenance_type,maintenance_status,downtime_hours,maintenance_cost
0,1,5,2026-07-25,Preventive,Completed,1.03,4534.96
1,2,2,2026-07-27,Corrective,Completed,5.40,8731.18
2,3,3,2026-07-27,Preventive,Completed,1.18,4547.77
3,4,2,2026-07-28,Routine Inspection,Completed,1.30,1758.16
4,5,4,2026-07-29,Routine Inspection,Completed,0.86,1860.77
5,6,5,2026-07-29,Routine Inspection,Completed,1.19,2066.30
6,7,1,2026-07-31,Corrective,Completed,5.38,9108.13
7,8,5,2026-08-03,Routine Inspection,In Progress,1.11,1903.05
8,9,4,2026-08-05,Routine Inspection,Completed,1.47,2078.70
9,10,1,2026-08-08,Preventive,In Progress,1.36,3048.84



TABLE: downtime


,downtime_id,machine_id,downtime_start,downtime_end,downtime_reason,downtime_hours
0,1,3,2026-07-20 06:00:00,2026-07-20 09:28:48,Mechanical Failure,3.48
1,2,2,2026-07-21 16:00:00,2026-07-21 18:08:24,Electrical Issue,2.14
2,3,4,2026-07-21 20:00:00,2026-07-21 21:01:12,Mechanical Failure,1.02
3,4,1,2026-07-22 13:00:00,2026-07-22 15:02:24,Mechanical Failure,2.04
4,5,3,2026-07-22 19:00:00,2026-07-22 19:34:12,Testing Error,0.57
5,6,1,2026-07-23 10:00:00,2026-07-23 11:06:00,Testing Error,1.10
6,7,2,2026-07-23 21:00:00,2026-07-23 21:46:12,Electrical Issue,0.77
7,8,2,2026-07-24 11:00:00,2026-07-24 14:13:48,Electrical Issue,3.23
8,9,3,2026-07-24 17:00:00,2026-07-24 18:27:36,Mechanical Failure,1.46
9,10,4,2026-07-24 12:00:00,2026-07-24 15:56:24,Machine Adjustment,3.94



TABLE: sensors


,sensor_id,machine_id,sensor_type,sensor_value,unit,recorded_at
0,1,1,Temperature,71.73,°C,2026-07-20 08:00:00
1,2,1,Vibration,1.80,mm/s,2026-07-20 08:00:00
2,3,1,Temperature,66.20,°C,2026-07-20 14:00:00
3,4,1,Vibration,3.02,mm/s,2026-07-20 14:00:00
4,5,1,Temperature,68.79,°C,2026-07-20 22:00:00
...,...,...,...,...,...,...
895,896,5,Vibration,3.50,mm/s,2026-08-18 08:00:00
896,897,5,Temperature,68.75,°C,2026-08-18 14:00:00
897,898,5,Vibration,2.78,mm/s,2026-08-18 14:00:00
898,899,5,Temperature,65.67,°C,2026-08-18 22:00:00


In [8]:
for table in important_tables:
    print("\n" + "=" * 60)
    print(f"NUMERIC SUMMARY: {table}")
    print("=" * 60)
    display(data[table].describe())


NUMERIC SUMMARY: machines


,machine_id
count,5.000000
mean,3.000000
std,1.581139
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,5.000000



NUMERIC SUMMARY: production


,production_id,machine_id,units_produced,units_rejected,production_time_hours
count,450.000000,450.000000,450.000000,450.000000,450.0
mean,225.500000,3.000000,911.237778,40.791111,8.0
std,130.048068,1.415788,73.708261,19.310057,0.0
min,1.000000,1.000000,670.000000,8.000000,8.0
25%,113.250000,2.000000,868.000000,24.000000,8.0
50%,225.500000,3.000000,921.000000,42.000000,8.0
75%,337.750000,4.000000,961.000000,58.750000,8.0
max,450.000000,5.000000,1134.000000,83.000000,8.0



NUMERIC SUMMARY: production_targets


...

,maintenance_id,equipment_id,downtime_hours,maintenance_cost
count,13.00000,13.000000,13.000000,13.000000
mean,7.00000,3.307692,2.122308,4771.213077
std,3.89444,1.493576,1.693636,3217.972525
min,1.00000,1.000000,0.860000,1758.160000
25%,4.00000,2.000000,1.110000,2066.300000
50%,7.00000,3.000000,1.300000,4534.960000
75%,10.00000,5.000000,2.040000,5924.240000
max,13.00000,5.000000,5.400000,11478.270000



NUMERIC SUMMARY: downtime


,downtime_id,machine_id,downtime_start,downtime_end,downtime_hours
count,32.000000,32.000000,32,32,32.000000
mean,16.500000,2.937500,2026-08-02 20:33:45,2026-08-02 22:46:36.750000,2.214375
min,1.000000,1.000000,2026-07-20 06:00:00,2026-07-20 09:28:48,0.570000
25%,8.750000,2.000000,2026-07-24 11:45:00,2026-07-24 15:30:45,1.335000
50%,16.500000,3.000000,2026-08-01 08:00:00,2026-08-01 11:15:18,2.150000
75%,24.250000,4.000000,2026-08-10 22:30:00,2026-08-10 23:32:06,3.177500
max,32.000000,5.000000,2026-08-18 13:00:00,2026-08-18 16:21:36,3.940000
std,9.380832,1.134147,NaN,NaN,1.047661



NUMERIC SUMMARY: sensors


,sensor_id,machine_id,sensor_value,recorded_at
count,900.000000,900.000,900.000000,900
mean,450.500000,3.000,36.894000,2026-08-04 02:40:00
min,1.000000,1.000,1.240000,2026-07-20 08:00:00
25%,225.750000,2.000,2.897500,2026-07-27 14:00:00
50%,450.500000,3.000,31.640000,2026-08-04 03:00:00
75%,675.250000,4.000,68.722500,2026-08-11 14:00:00
max,900.000000,5.000,88.870000,2026-08-18 22:00:00
std,259.951919,1.415,34.034031,NaN


In [9]:
for table in important_tables:
    print("\n" + "=" * 60)
    print(f"CATEGORICAL VALUES: {table}")
    print("=" * 60)

    for column in data[table].select_dtypes(include=["object", "str"]).columns:
        print(f"\n{column}:")
        print(data[table][column].unique())


CATEGORICAL VALUES: machines

machine_name:
<StringArray>
['PCB Assembly Line 01', 'PCB Assembly Line 02',   'CNC Precision Unit',
   'Testing Station 01',   'Testing Station 02']
Length: 5, dtype: str

machine_type:
<StringArray>
['Assembly Line', 'CNC Machine', 'Testing Equipment']
Length: 3, dtype: str

location:
<StringArray>
['Production Floor A', 'Production Floor B', 'Quality Floor']
Length: 3, dtype: str

status:
<StringArray>
['Operational', 'Maintenance']
Length: 2, dtype: str

installation_date:
[datetime.date(2022, 5, 10) datetime.date(2022, 8, 15)
 datetime.date(2021, 3, 20) datetime.date(2023, 1, 12)
 datetime.date(2023, 4, 18)]

CATEGORICAL VALUES: production

production_date:
[datetime.date(2026, 7, 20) datetime.date(2026, 7, 21)
 datetime.date(2026, 7, 22) datetime.date(2026, 7, 23)
 datetime.date(2026, 7, 24) datetime.date(2026, 7, 25)
 datetime.date(2026, 7, 26) datetime.date(2026, 7, 27)
 datetime.date(2026, 7, 28) datetime.date(2026, 7, 29)
 datetime.date(2026, 7,

In [10]:
print("Machines referenced in production:")
print(sorted(data["production"]["machine_id"].unique()))

print("\nMachines in machines table:")
print(sorted(data["machines"]["machine_id"].unique()))

print("\nProduction IDs referenced in quality:")
print(sorted(data["quality"]["production_id"].unique()))

print("\nProduction IDs in production table:")
print(sorted(data["production"]["production_id"].unique()))

print("\nMachines referenced in downtime:")
print(sorted(data["downtime"]["machine_id"].unique()))

print("\nMachines referenced in sensors:")
print(sorted(data["sensors"]["machine_id"].unique()))

Machines referenced in production:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Machines in machines table:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Production IDs referenced in quality:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.

In [11]:
# ============================================================
# DATA CLEANING PIPELINE
# CHECK FOR  MISSING VALUES
# ============================================================

print("MISSING VALUE CHECK")
print("=" * 70)

for table, df in data.items():
    print(f"\nTABLE: {table}")
    print("-" * 50)

    missing = df.isnull().sum()

    if missing.sum() == 0:
        print("No missing values found.")
    else:
        print(missing[missing > 0])

MISSING VALUE CHECK

TABLE: downtime
--------------------------------------------------
No missing values found.

TABLE: employees
--------------------------------------------------
No missing values found.

TABLE: inventory
--------------------------------------------------
No missing values found.

TABLE: machines
--------------------------------------------------
No missing values found.

TABLE: maintenance
--------------------------------------------------
No missing values found.

TABLE: production
--------------------------------------------------
No missing values found.

TABLE: production_logs
--------------------------------------------------
No missing values found.

TABLE: production_targets
--------------------------------------------------
No missing values found.

TABLE: quality
--------------------------------------------------
No missing values found.

TABLE: sensors
--------------------------------------------------
No missing values found.

TABLE: shifts
-------------

In [12]:
# ============================================================
# CHECK DUPLICATE ROWS
# ============================================================

print("DUPLICATE ROW CHECK")
print("=" * 70)

for table, df in data.items():
    duplicate_count = df.duplicated().sum()

    print(f"{table}: {duplicate_count} duplicate rows")

DUPLICATE ROW CHECK
downtime: 0 duplicate rows
employees: 0 duplicate rows
inventory: 0 duplicate rows
machines: 0 duplicate rows
maintenance: 0 duplicate rows
production: 0 duplicate rows
production_logs: 0 duplicate rows
production_targets: 0 duplicate rows
quality: 0 duplicate rows
sensors: 0 duplicate rows
shifts: 0 duplicate rows
suppliers: 0 duplicate rows


In [13]:




# CHECK INVALID NUMERIC VALUES
# ============================================================

print("INVALID NUMERIC VALUE CHECK")
print("=" * 70)

for table, df in data.items():

    print(f"\nTABLE: {table}")
    print("-" * 50)

    found_invalid = False

    for col in df.columns:

        # Skip datetime columns
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue

        # Skip timedelta columns
        if pd.api.types.is_timedelta64_dtype(df[col]):
            continue

        # Check only actual numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            negative_count = (df[col] < 0).sum()

            if negative_count > 0:
                print(f"{col}: {negative_count} negative values")
                found_invalid = True

    if not found_invalid:
        print("No negative numeric values found.")

INVALID NUMERIC VALUE CHECK

TABLE: downtime
--------------------------------------------------
No negative numeric values found.

TABLE: employees
--------------------------------------------------
No negative numeric values found.

TABLE: inventory
--------------------------------------------------
No negative numeric values found.

TABLE: machines
--------------------------------------------------
No negative numeric values found.

TABLE: maintenance
--------------------------------------------------
No negative numeric values found.

TABLE: production
--------------------------------------------------
No negative numeric values found.

TABLE: production_logs
--------------------------------------------------
No negative numeric values found.

TABLE: production_targets
--------------------------------------------------
No negative numeric values found.

TABLE: quality
--------------------------------------------------
No negative numeric values found.

TABLE: sensors
---------------

In [14]:

# DATE/TIME VALIDATION


print("DATE/TIME VALIDATION")
print("=" * 70)

for table, df in data.items():

    datetime_cols = df.select_dtypes(
        include=["datetime", "datetimetz"]
    ).columns

    print(f"\nTABLE: {table}")
    print("-" * 50)

    if len(datetime_cols) == 0:
        print("No datetime columns found.")
        continue

    for col in datetime_cols:
        print(f"{col}:")
        print(f"  Minimum: {df[col].min()}")
        print(f"  Maximum: {df[col].max()}")
        print(f"  Missing: {df[col].isna().sum()}")

DATE/TIME VALIDATION

TABLE: downtime
--------------------------------------------------
downtime_start:
  Minimum: 2026-07-20 06:00:00
  Maximum: 2026-08-18 13:00:00
  Missing: 0
downtime_end:
  Minimum: 2026-07-20 09:28:48
  Maximum: 2026-08-18 16:21:36
  Missing: 0

TABLE: employees
--------------------------------------------------
No datetime columns found.

TABLE: inventory
--------------------------------------------------
No datetime columns found.

TABLE: machines
--------------------------------------------------
No datetime columns found.

TABLE: maintenance
--------------------------------------------------
No datetime columns found.

TABLE: production
--------------------------------------------------
No datetime columns found.

TABLE: production_logs
--------------------------------------------------
log_date:
  Minimum: 2026-07-20 00:00:00
  Maximum: 2026-08-18 00:00:00
  Missing: 0

TABLE: production_targets
--------------------------------------------------
No datetime

In [15]:
print("BUSINESS RULE VALIDATION")
print("=" * 70)

# Production checks
production = data["production"]

print("\nPRODUCTION")
print("-" * 50)

if "units_produced" in production.columns:
    print("Negative production:",
          (production["units_produced"] < 0).sum())

if "target_quantity" in production.columns:
    print("Negative target quantity:",
          (production["target_quantity"] < 0).sum())

# Quality checks
quality = data["quality"]

print("\nQUALITY")
print("-" * 50)

if "defect_quantity" in quality.columns:
    print("Negative defects:",
          (quality["defect_quantity"] < 0).sum())

# Downtime checks
downtime = data["downtime"]

print("\nDOWNTIME")
print("-" * 50)

if "downtime_start" in downtime.columns and "downtime_end" in downtime.columns:
    invalid_downtime = (
        downtime["downtime_end"] < downtime["downtime_start"]
    ).sum()

    print("Invalid downtime periods:", invalid_downtime)

# Production target checks
targets = data["production_targets"]

print("\nPRODUCTION TARGETS")
print("-" * 50)

numeric_target_cols = targets.select_dtypes(include="number").columns

for col in numeric_target_cols:
    print(f"{col} negative values:", (targets[col] < 0).sum())

BUSINESS RULE VALIDATION

PRODUCTION
--------------------------------------------------
Negative production: 0

QUALITY
--------------------------------------------------

DOWNTIME
--------------------------------------------------
Invalid downtime periods: 0

PRODUCTION TARGETS
--------------------------------------------------
target_id negative values: 0
machine_id negative values: 0
target_quantity negative values: 0


In [16]:
print("DATA TYPE VALIDATION")
print("=" * 70)

for table, df in data.items():

    print(f"\nTABLE: {table}")
    print("-" * 50)

    for column in df.columns:
        print(f"{column}: {df[column].dtype}")

DATA TYPE VALIDATION

TABLE: downtime
--------------------------------------------------
downtime_id: int64
machine_id: int64
downtime_start: datetime64[us]
downtime_end: datetime64[us]
downtime_reason: str
downtime_hours: float64

TABLE: employees
--------------------------------------------------
employee_id: int64
employee_name: str
department: str
role: str
shift: str

TABLE: inventory
--------------------------------------------------
inventory_id: int64
material_name: str
quantity_available: int64
reorder_level: int64
unit: str
last_updated: object

TABLE: machines
--------------------------------------------------
machine_id: int64
machine_name: str
machine_type: str
location: str
status: str
installation_date: object

TABLE: maintenance
--------------------------------------------------
maintenance_id: int64
equipment_id: int64
maintenance_date: object
maintenance_type: str
maintenance_status: str
downtime_hours: float64
maintenance_cost: float64

TABLE: production
------------

In [17]:
print("OUTLIER CHECK")
print("=" * 70)

for table, df in data.items():

    numeric_cols = df.select_dtypes(include=["number"]).columns

    print(f"\nTABLE: {table}")
    print("-" * 50)

    if len(numeric_cols) == 0:
        print("No numeric columns found.")
        continue

    for col in numeric_cols:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = (
            (df[col] < lower_bound) |
            (df[col] > upper_bound)
        ).sum()

        print(f"{col}: {outliers} potential outliers")

OUTLIER CHECK

TABLE: downtime
--------------------------------------------------
downtime_id: 0 potential outliers
machine_id: 0 potential outliers
downtime_hours: 0 potential outliers

TABLE: employees
--------------------------------------------------
employee_id: 0 potential outliers

TABLE: inventory
--------------------------------------------------
inventory_id: 0 potential outliers
quantity_available: 0 potential outliers
reorder_level: 0 potential outliers

TABLE: machines
--------------------------------------------------
machine_id: 0 potential outliers

TABLE: maintenance
--------------------------------------------------
maintenance_id: 0 potential outliers
equipment_id: 0 potential outliers
downtime_hours: 3 potential outliers
maintenance_cost: 0 potential outliers

TABLE: production
--------------------------------------------------
production_id: 0 potential outliers
machine_id: 0 potential outliers
units_produced: 6 potential outliers
units_rejected: 0 potential outlie

In [18]:
 PRODUCTION EDA

production = data["production"]

print("PRODUCTION DATA")
print("=" * 70)

print("\nShape:")
print(production.shape)

print("\nColumns:")
print(production.columns.tolist())

print("\nFirst 5 rows:")
display(production.head())

print("\nStatistical Summary:")
display(production.describe())

IndentationError: unexpected indent (2182687487.py, line 1)

In [ ]:
# PRODUCTION ANALYSIS

print("PRODUCTION ANALYSIS")
print("=" * 70)

# Total production
print("\nTotal Units Produced:")
print(production["units_produced"].sum())

# Average production
print("\nAverage Units Produced:")
print(production["units_produced"].mean())

# Minimum production
print("\nMinimum Units Produced:")
print(production["units_produced"].min())

# Maximum production
print("\nMaximum Units Produced:")
print(production["units_produced"].max())

# Machine-wise production
print("\nMachine-wise Production:")
machine_production = (
    production.groupby("machine_id")["units_produced"]
    .agg(["sum", "mean", "min", "max"])
    .reset_index()
)

display(machine_production)

PRODUCTION ANALYSIS

Total Units Produced:
410057

Average Units Produced:
911.2377777777778

Minimum Units Produced:
670

Maximum Units Produced:
1134

Machine-wise Production:


,machine_id,sum,mean,min,max
0,1,84982,944.244444,842,1034
1,2,81096,901.066667,817,973
2,3,71938,799.311111,670,902
3,4,88769,986.322222,903,1134
4,5,83272,925.244444,845,1022


In [ ]:
# SHIFT-WISE PRODUCTION EDA

print("SHIFT-WISE PRODUCTION ANALYSIS")
print("=" * 70)

shift_production = (
    production.groupby("shift")["units_produced"]
    .agg(["sum", "mean", "min", "max"])
    .reset_index()
)

display(shift_production)

SHIFT-WISE PRODUCTION ANALYSIS


,shift,sum,mean,min,max
0,Afternoon,136822,912.146667,711,1066
1,Morning,136997,913.313333,723,1134
2,Night,136238,908.253333,670,1070


In [ ]:
# TARGET VS ACTUAL PRODUCTION

print("TARGET VS ACTUAL PRODUCTION")
print("=" * 70)

targets = data["production_targets"]

print("\nTarget Data:")
print("Shape:", targets.shape)
print("Columns:", targets.columns.tolist())

display(targets.head())

TARGET VS ACTUAL PRODUCTION

Target Data:
Shape: (450, 4)
Columns: ['target_id', 'machine_id', 'target_date', 'target_quantity']


,target_id,machine_id,target_date,target_quantity
0,1,1,2026-07-20,998
1,2,1,2026-07-20,1012
2,3,1,2026-07-20,1003
3,4,2,2026-07-20,873
4,5,2,2026-07-20,907


In [ ]:
targets = data["production_targets"]

print("TARGET TABLE")
print("=" * 70)
print("Shape:", targets.shape)
print("Columns:", targets.columns.tolist())
print("\nFirst 5 rows:")
print(targets.head().to_string())

TARGET TABLE
Shape: (450, 4)
Columns: ['target_id', 'machine_id', 'target_date', 'target_quantity']

First 5 rows:
   target_id  machine_id target_date  target_quantity
0          1           1  2026-07-20              998
1          2           1  2026-07-20             1012
2          3           1  2026-07-20             1003
3          4           2  2026-07-20              873
4          5           2  2026-07-20              907


In [ ]:
# TARGET VS ACTUAL PRODUCTION

target_vs_actual = production.merge(
    targets,
    left_on=["machine_id", "production_date"],
    right_on=["machine_id", "target_date"],
    how="left"
)

target_vs_actual["achievement_percent"] = (
    target_vs_actual["units_produced"]
    / target_vs_actual["target_quantity"]
) * 100

target_vs_actual["difference"] = (
    target_vs_actual["units_produced"]
    - target_vs_actual["target_quantity"]
)

print("TARGET VS ACTUAL PRODUCTION")
print("=" * 70)

print("\nOverall Target Quantity:")
print(target_vs_actual["target_quantity"].sum())

print("\nOverall Actual Production:")
print(target_vs_actual["units_produced"].sum())

print("\nOverall Achievement %:")
print(
    target_vs_actual["units_produced"].sum()
    / target_vs_actual["target_quantity"].sum()
    * 100
)

print("\nMachine-wise Target vs Actual:")
machine_target_actual = (
    target_vs_actual
    .groupby("machine_id")
    .agg(
        target_quantity=("target_quantity", "sum"),
        actual_production=("units_produced", "sum")
    )
    .reset_index()
)

machine_target_actual["achievement_percent"] = (
    machine_target_actual["actual_production"]
    / machine_target_actual["target_quantity"]
) * 100

machine_target_actual["difference"] = (
    machine_target_actual["actual_production"]
    - machine_target_actual["target_quantity"]
)

print(machine_target_actual.to_string(index=False))

TARGET VS ACTUAL PRODUCTION

Overall Target Quantity:
1258032

Overall Actual Production:
1230171

Overall Achievement %:
97.78535045213476

Machine-wise Target vs Actual:
 machine_id  target_quantity  actual_production  achievement_percent  difference
          1           271278             254946            93.979608      -16332
          2           242712             243288           100.237318         576
          3           216387             215814            99.735197        -573
          4           270837             266307            98.327407       -4530
          5           256818             249816            97.273556       -7002


In [19]:
# QUALITY / DEFECT EDA

quality = data["quality"]

print("QUALITY DATA")
print("=" * 70)

print("Shape:", quality.shape)
print("Columns:", quality.columns.tolist())

print("\nFirst 5 rows:")
print(quality.head().to_string())

print("\nStatistical Summary:")
print(quality.describe().to_string())

QUALITY DATA
Shape: (450, 6)
Columns: ['quality_id', 'production_id', 'inspection_date', 'defect_type', 'defect_count', 'quality_status']

First 5 rows:
   quality_id  production_id inspection_date             defect_type  defect_count quality_status
0           1              1      2026-07-20              PCB Damage            17         Fail\r
1           2              2      2026-07-20             Overheating             9         Pass\r
2           3              3      2026-07-20           Solder Defect             3         Pass\r
3           4              4      2026-07-20              PCB Damage            24         Fail\r
4           5              5      2026-07-20  Component Misplacement             3         Pass\r

Statistical Summary:
       quality_id  production_id  defect_count
count  450.000000     450.000000    450.000000
mean   225.500000     225.500000     10.522222
std    130.048068     130.048068      6.258051
min      1.000000       1.000000      1.000000
25

In [20]:
#  QUALITY STATUS ANALYSIS

print("QUALITY STATUS ANALYSIS")
print("=" * 70)

print("\nQuality Status Counts:")
print(quality["quality_status"].value_counts().to_string())

print("\nQuality Status Percentage:")
print(
    (quality["quality_status"].value_counts(normalize=True) * 100)
    .round(2)
    .to_string()
)

QUALITY STATUS ANALYSIS

Quality Status Counts:
quality_status
Pass\r    332
Fail\r    118

Quality Status Percentage:
quality_status
Pass\r    73.78
Fail\r    26.22


In [21]:
# DEFECT TYPE ANALYSIS

print("DEFECT TYPE ANALYSIS")
print("=" * 70)

defect_analysis = (
    quality.groupby("defect_type")["defect_count"]
    .agg(["count", "sum", "mean", "min", "max"])
    .sort_values("sum", ascending=False)
    .reset_index()
)

print("\nDefect Type Summary:")
print(defect_analysis.to_string(index=False))

DEFECT TYPE ANALYSIS

Defect Type Summary:
           defect_type  count  sum      mean  min  max
     Missing Component    110 1139 10.354545    1   27
         Solder Defect     97 1079 11.123711    2   24
Component Misplacement     74  892 12.054054    2   26
            PCB Damage     85  871 10.247059    1   24
           Overheating     84  754  8.976190    1   26


In [22]:
# STAGE 17.5.3 — MACHINE-WISE QUALITY ANALYSIS

print("MACHINE-WISE QUALITY ANALYSIS")
print("=" * 70)

# Connect quality data with production data to get machine_id
quality_machine = quality.merge(
    production[["production_id", "machine_id"]],
    on="production_id",
    how="left"
)

machine_quality = (
    quality_machine.groupby("machine_id")
    .agg(
        total_defects=("defect_count", "sum"),
        average_defects=("defect_count", "mean"),
        inspections=("quality_id", "count")
    )
    .reset_index()
)

machine_quality["defects_per_inspection"] = (
    machine_quality["total_defects"]
    / machine_quality["inspections"]
)

print("\nMachine-wise Quality Summary:")
print(machine_quality.to_string(index=False))

MACHINE-WISE QUALITY ANALYSIS

Machine-wise Quality Summary:
 machine_id  total_defects  average_defects  inspections  defects_per_inspection
          1            948        10.533333           90               10.533333
          2           1002        11.133333           90               11.133333
          3            854         9.488889           90                9.488889
          4           1101        12.233333           90               12.233333
          5            830         9.222222           90                9.222222


In [23]:
#   SENSOR EDA

sensors = data["sensors"]

print("SENSOR DATA")
print("=" * 70)

print("Shape:", sensors.shape)
print("Columns:", sensors.columns.tolist())

print("\nFirst 5 rows:")
print(sensors.head().to_string())

print("\nStatistical Summary:")
print(sensors.describe().to_string())

SENSOR DATA
Shape: (900, 6)
Columns: ['sensor_id', 'machine_id', 'sensor_type', 'sensor_value', 'unit', 'recorded_at']

First 5 rows:
   sensor_id  machine_id  sensor_type  sensor_value  unit         recorded_at
0          1           1  Temperature         71.73    °C 2026-07-20 08:00:00
1          2           1    Vibration          1.80  mm/s 2026-07-20 08:00:00
2          3           1  Temperature         66.20    °C 2026-07-20 14:00:00
3          4           1    Vibration          3.02  mm/s 2026-07-20 14:00:00
4          5           1  Temperature         68.79    °C 2026-07-20 22:00:00

Statistical Summary:
        sensor_id  machine_id  sensor_value          recorded_at
count  900.000000     900.000    900.000000                  900
mean   450.500000       3.000     36.894000  2026-08-04 02:40:00
min      1.000000       1.000      1.240000  2026-07-20 08:00:00
25%    225.750000       2.000      2.897500  2026-07-27 14:00:00
50%    450.500000       3.000     31.640000  2026-0

In [24]:
#  SENSOR TYPE ANALYSIS

print("SENSOR TYPE ANALYSIS")
print("=" * 70)

sensor_type_summary = (
    sensors.groupby("sensor_type")["sensor_value"]
    .agg(["count", "mean", "min", "max", "std"])
    .reset_index()
)

print("\nSensor Type Summary:")
print(sensor_type_summary.to_string(index=False))

SENSOR TYPE ANALYSIS

Sensor Type Summary:
sensor_type  count      mean   min   max      std
Temperature    450 70.566333 57.23 88.87 6.737070
  Vibration    450  3.221667  1.24  6.05 1.058174


In [25]:
# MACHINE-WISE SENSOR ANALYSIS

print("MACHINE-WISE SENSOR ANALYSIS")
print("=" * 70)

machine_sensor_summary = (
    sensors.groupby(["machine_id", "sensor_type"])["sensor_value"]
    .agg(["count", "mean", "min", "max", "std"])
    .reset_index()
)

print("\nMachine-wise Sensor Summary:")
print(machine_sensor_summary.to_string(index=False))

MACHINE-WISE SENSOR ANALYSIS

Machine-wise Sensor Summary:
 machine_id sensor_type  count      mean   min   max      std
          1 Temperature     90 69.953222 61.23 76.56 3.333519
          1   Vibration     90  3.005111  1.31  4.22 0.545425
          2 Temperature     90 68.170333 60.45 75.40 3.001107
          2   Vibration     90  2.765222  1.60  3.95 0.472462
          3 Temperature     90 82.204000 73.72 88.87 3.052608
          3   Vibration     90  5.074000  3.76  6.05 0.450674
          4 Temperature     90 65.419111 57.23 75.49 2.914622
          4   Vibration     90  2.547667  1.24  3.78 0.468955
          5 Temperature     90 67.085000 60.50 75.02 2.976255
          5   Vibration     90  2.716333  1.60  3.90 0.509746


In [26]:
# SENSOR TREND ANALYSIS

print("SENSOR TREND ANALYSIS")
print("=" * 70)

sensor_trends = (
    sensors.groupby(
        [sensors["recorded_at"].dt.date, "sensor_type"]
    )["sensor_value"]
    .agg(["mean", "min", "max"])
    .reset_index()
)

print("\nDaily Sensor Trends:")
print(sensor_trends.to_string(index=False))

SENSOR TREND ANALYSIS

Daily Sensor Trends:
recorded_at sensor_type      mean   min   max
 2026-07-20 Temperature 69.601333 61.66 82.82
 2026-07-20   Vibration  3.079333  1.74  5.60
 2026-07-21 Temperature 69.848667 61.28 85.19
 2026-07-21   Vibration  3.086667  2.25  5.28
 2026-07-22 Temperature 71.417333 62.27 82.98
 2026-07-22   Vibration  3.202667  1.74  5.81
 2026-07-23 Temperature 71.374667 63.40 88.72
 2026-07-23   Vibration  3.170000  1.98  5.23
 2026-07-24 Temperature 71.696667 60.57 83.34
 2026-07-24   Vibration  3.184667  1.99  5.51
 2026-07-25 Temperature 69.204667 60.78 79.39
 2026-07-25   Vibration  3.356667  2.17  5.47
 2026-07-26 Temperature 70.712667 62.01 83.19
 2026-07-26   Vibration  3.042667  2.01  5.08
 2026-07-27 Temperature 71.728667 62.42 85.01
 2026-07-27   Vibration  3.316000  2.30  5.74
 2026-07-28 Temperature 69.956000 63.43 87.65
 2026-07-28   Vibration  3.387333  2.12  5.96
 2026-07-29 Temperature 71.522000 61.93 84.71
 2026-07-29   Vibration  3.175333  1

In [27]:
# DOWNTIME EDA

downtime = data["downtime"]

print("DOWNTIME DATA")
print("=" * 70)

print("Shape:", downtime.shape)
print("Columns:", downtime.columns.tolist())

print("\nFirst 5 rows:")
print(downtime.head().to_string())

print("\nStatistical Summary:")
print(downtime.describe().to_string())

DOWNTIME DATA
Shape: (32, 6)
Columns: ['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']

First 5 rows:
   downtime_id  machine_id      downtime_start        downtime_end     downtime_reason  downtime_hours
0            1           3 2026-07-20 06:00:00 2026-07-20 09:28:48  Mechanical Failure            3.48
1            2           2 2026-07-21 16:00:00 2026-07-21 18:08:24    Electrical Issue            2.14
2            3           4 2026-07-21 20:00:00 2026-07-21 21:01:12  Mechanical Failure            1.02
3            4           1 2026-07-22 13:00:00 2026-07-22 15:02:24  Mechanical Failure            2.04
4            5           3 2026-07-22 19:00:00 2026-07-22 19:34:12       Testing Error            0.57

Statistical Summary:
       downtime_id  machine_id       downtime_start                downtime_end  downtime_hours
count    32.000000   32.000000                   32                          32       32.000000
mean     16.50

In [28]:
# DOWNTIME REASON ANALYSIS

print("DOWNTIME REASON ANALYSIS")
print("=" * 70)

downtime_reason = (
    downtime.groupby("downtime_reason")["downtime_hours"]
    .agg(["count", "sum", "mean", "min", "max"])
    .sort_values("sum", ascending=False)
    .reset_index()
)

print("\nDowntime Reason Summary:")
print(downtime_reason.to_string(index=False))

DOWNTIME REASON ANALYSIS

Downtime Reason Summary:
   downtime_reason  count   sum     mean  min  max
Mechanical Failure      8 18.54 2.317500 1.02 3.77
  Electrical Issue      8 15.08 1.885000 0.67 3.23
Machine Adjustment      5 14.21 2.842000 1.60 3.94
       Calibration      3  9.35 3.116667 2.20 3.66
     Testing Error      6  9.13 1.521667 0.57 2.40
 Material Shortage      2  4.55 2.275000 0.94 3.61


In [29]:
# MACHINE-WISE DOWNTIME ANALYSIS

print("MACHINE-WISE DOWNTIME ANALYSIS")
print("=" * 70)

machine_downtime = (
    downtime.groupby("machine_id")["downtime_hours"]
    .agg(["count", "sum", "mean", "min", "max"])
    .reset_index()
)

machine_downtime.columns = [
    "machine_id",
    "downtime_events",
    "total_downtime_hours",
    "average_downtime_hours",
    "minimum_downtime_hours",
    "maximum_downtime_hours"
]

print("\nMachine-wise Downtime Summary:")
print(machine_downtime.to_string(index=False))

MACHINE-WISE DOWNTIME ANALYSIS

Machine-wise Downtime Summary:
 machine_id  downtime_events  total_downtime_hours  average_downtime_hours  minimum_downtime_hours  maximum_downtime_hours
          1                4                  7.06                1.765000                    1.10                    2.15
          2                6                 15.49                2.581667                    0.77                    3.66
          3               13                 26.39                2.030000                    0.57                    3.77
          4                6                 16.30                2.716667                    1.02                    3.94
          5                3                  5.62                1.873333                    0.94                    3.36


In [30]:
#  MAINTENANCE EDA

maintenance = data["maintenance"]

print("MAINTENANCE DATA")
print("=" * 70)

print("Shape:", maintenance.shape)
print("Columns:", maintenance.columns.tolist())

print("\nFirst 5 rows:")
print(maintenance.head().to_string())

print("\nStatistical Summary:")
print(maintenance.describe().to_string())

MAINTENANCE DATA
Shape: (13, 7)
Columns: ['maintenance_id', 'equipment_id', 'maintenance_date', 'maintenance_type', 'maintenance_status', 'downtime_hours', 'maintenance_cost']

First 5 rows:
   maintenance_id  equipment_id maintenance_date    maintenance_type maintenance_status  downtime_hours  maintenance_cost
0               1             5       2026-07-25          Preventive          Completed            1.03           4534.96
1               2             2       2026-07-27          Corrective          Completed            5.40           8731.18
2               3             3       2026-07-27          Preventive          Completed            1.18           4547.77
3               4             2       2026-07-28  Routine Inspection          Completed            1.30           1758.16
4               5             4       2026-07-29  Routine Inspection          Completed            0.86           1860.77

Statistical Summary:
       maintenance_id  equipment_id  downtime_hours  ma

In [31]:
#  MAINTENANCE TYPE ANALYSIS

print("MAINTENANCE TYPE ANALYSIS")
print("=" * 70)

maintenance_type_summary = (
    maintenance.groupby("maintenance_type")
    .agg(
        maintenance_count=("maintenance_id", "count"),
        total_downtime=("downtime_hours", "sum"),
        avg_downtime=("downtime_hours", "mean"),
        total_cost=("maintenance_cost", "sum"),
        avg_cost=("maintenance_cost", "mean")
    )
    .reset_index()
)

print(maintenance_type_summary.to_string(index=False))

MAINTENANCE TYPE ANALYSIS
  maintenance_type  maintenance_count  total_downtime  avg_downtime  total_cost    avg_cost
        Corrective                  3           15.04      5.013333    29317.58 9772.526667
        Preventive                  5            6.62      1.324000    23041.21 4608.242000
Routine Inspection                  5            5.93      1.186000     9666.98 1933.396000


In [32]:
# MACHINE-WISE MAINTENANCE ANALYSIS

print("MACHINE-WISE MAINTENANCE ANALYSIS")
print("=" * 70)

machine_maintenance = (
    maintenance.groupby("equipment_id")
    .agg(
        maintenance_count=("maintenance_id", "count"),
        total_downtime=("downtime_hours", "sum"),
        total_cost=("maintenance_cost", "sum")
    )
    .reset_index()
)

print(machine_maintenance.to_string(index=False))

MACHINE-WISE MAINTENANCE ANALYSIS
 equipment_id  maintenance_count  total_downtime  total_cost
            1                  2            6.74    12156.97
            2                  2            6.70    10489.34
            3                  3            7.48    21011.44
            4                  2            2.33     3939.47
            5                  4            4.34    14428.55


In [33]:
#  CORRELATION ANALYSIS

print("CORRELATION ANALYSIS")
print("=" * 70)

# Production + quality + machine information
analysis_df = production.merge(
    quality[["production_id", "defect_count"]],
    on="production_id",
    how="left"
)

# Correlation between important numerical variables
correlation = analysis_df[
    ["units_produced", "units_rejected", "production_time_hours", "defect_count"]
].corr()

print("\nCorrelation Matrix:")
print(correlation.round(2).to_string())

CORRELATION ANALYSIS

Correlation Matrix:
                       units_produced  units_rejected  production_time_hours  defect_count
units_produced                   1.00            0.20                    NaN          0.14
units_rejected                   0.20            1.00                    NaN          0.84
production_time_hours             NaN             NaN                    NaN           NaN
defect_count                     0.14            0.84                    NaN          1.00


In [34]:
#  EDA CONCLUSIONS

print("EDA CONCLUSIONS")
print("=" * 70)

print("""
1. PRODUCTION:
   - Average production is approximately 911 units per record.
   - Morning shift has the highest average production.
   - Night shift has the lowest average production, but the difference is small.

2. TARGET ACHIEVEMENT:
   - Actual production is compared against machine-wise production targets.
   - Target achievement will be used as an important KPI and ML feature.

3. QUALITY:
   - 73.78% of quality inspections passed.
   - 26.22% of inspections failed.
   - Missing Component has the highest total defect count.
   - Machine 4 has the highest average defects.

4. SENSORS:
   - Average temperature is approximately 70.57°C.
   - Average vibration is approximately 3.22 mm/s.
   - Machine 3 has the highest average temperature and vibration.
   - Sensor readings vary across machines and over time.

5. DOWNTIME:
   - There are 32 downtime events.
   - Average downtime per event is approximately 2.21 hours.
   - Mechanical Failure contributes the highest total downtime.
   - Machine 3 has the highest total downtime.

6. MAINTENANCE:
   - Corrective maintenance has the highest downtime and cost per event.
   - Machine 3 has the highest maintenance downtime and maintenance cost.

7. CORRELATION:
   - Units rejected and defect count have a strong positive correlation (0.84).
   - Units produced and defect count have only a weak correlation.
   - Production time is constant at 8 hours, so it has no meaningful correlation.

8. ML RELEVANCE:
   - Machine 3 shows higher sensor values, downtime and maintenance activity.
   - Sensor readings, quality metrics, downtime and maintenance history
     can therefore be considered useful inputs for future ML modeling.
""")

EDA CONCLUSIONS

1. PRODUCTION:
   - Average production is approximately 911 units per record.
   - Morning shift has the highest average production.
   - Night shift has the lowest average production, but the difference is small.

2. TARGET ACHIEVEMENT:
   - Actual production is compared against machine-wise production targets.
   - Target achievement will be used as an important KPI and ML feature.

3. QUALITY:
   - 73.78% of quality inspections passed.
   - 26.22% of inspections failed.
   - Missing Component has the highest total defect count.
   - Machine 4 has the highest average defects.

4. SENSORS:
   - Average temperature is approximately 70.57°C.
   - Average vibration is approximately 3.22 mm/s.
   - Machine 3 has the highest average temperature and vibration.
   - Sensor readings vary across machines and over time.

5. DOWNTIME:
   - There are 32 downtime events.
   - Average downtime per event is approximately 2.21 hours.
   - Mechanical Failure contributes the highest to

In [35]:
#  PRODUCTION KPIs

print("PRODUCTION KPIs")
print("=" * 70)

total_production = production["units_produced"].sum()
total_rejected = production["units_rejected"].sum()

total_output = total_production + total_rejected

production_rejection_rate = (
    total_rejected / total_output
) * 100

print("\nTotal Units Produced:")
print(total_production)

print("\nTotal Units Rejected:")
print(total_rejected)

print("\nTotal Production Output:")
print(total_output)

print("\nRejection Rate (%):")
print(round(production_rejection_rate, 2))

PRODUCTION KPIs

Total Units Produced:
410057

Total Units Rejected:
18356

Total Production Output:
428413

Rejection Rate (%):
4.28


In [36]:
print("TARGET ACHIEVEMENT KPIs")
print("=" * 70)

total_produced = production["units_produced"].sum()
total_target = production_targets["target_quantity"].sum()

overall_target_achievement = (
    total_produced / total_target
) * 100

print("\nTotal Target Quantity:")
print(total_target)

print("\nTotal Units Produced:")
print(total_produced)

print("\nOverall Target Achievement (%):")
print(round(overall_target_achievement, 2))

TARGET ACHIEVEMENT KPIs


NameError: name 'production_targets' is not defined

In [37]:
print([name for name in globals() if not name.startswith("_")])

['In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'pd', 'mysql', 'sys', 'os', 'get_connection', 'connection', 'cursor', 'tables', 'table', 'data', 'query', 'df', 'summary', 'summary_df', 'important_tables', 'column', 'missing', 'duplicate_count', 'found_invalid', 'col', 'negative_count', 'datetime_cols', 'production', 'quality', 'downtime', 'invalid_downtime', 'targets', 'numeric_target_cols', 'numeric_cols', 'Q1', 'Q3', 'IQR', 'lower_bound', 'upper_bound', 'outliers', 'defect_analysis', 'quality_machine', 'machine_quality', 'sensors', 'sensor_type_summary', 'machine_sensor_summary', 'sensor_trends', 'downtime_reason', 'machine_downtime', 'maintenance', 'maintenance_type_summary', 'machine_maintenance', 'analysis_df', 'correlation', 'total_production', 'total_rejected', 'total_output', 'production_rejection_rate', 'total_produced']


In [38]:
print("TARGET ACHIEVEMENT KPIs")
print("=" * 70)

total_produced = production["units_produced"].sum()
total_target = targets["target_quantity"].sum()

overall_target_achievement = (
    total_produced / total_target
) * 100

print("\nTotal Target Quantity:")
print(total_target)

print("\nTotal Units Produced:")
print(total_produced)

print("\nOverall Target Achievement (%):")
print(round(overall_target_achievement, 2))

TARGET ACHIEVEMENT KPIs

Total Target Quantity:
419344

Total Units Produced:
410057

Overall Target Achievement (%):
97.79


In [39]:
print("PRODUCTION COLUMNS:")
print(production.columns.tolist())

print("\nTARGET COLUMNS:")
print(targets.columns.tolist())

PRODUCTION COLUMNS:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours']

TARGET COLUMNS:
['target_id', 'machine_id', 'target_date', 'target_quantity']


In [40]:
print("ACTUAL PRODUCTION TABLE COLUMNS")
print("=" * 70)

cursor.execute("DESCRIBE production")

for row in cursor.fetchall():
    print(row)

ACTUAL PRODUCTION TABLE COLUMNS
('production_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('machine_id', 'int', 'NO', 'MUL', None, '')
('production_date', 'date', 'NO', '', None, '')
('shift', 'varchar(20)', 'YES', '', None, '')
('units_produced', 'int', 'YES', '', None, '')
('units_rejected', 'int', 'YES', '', None, '')
('production_time_hours', 'decimal(5,2)', 'YES', '', None, '')


In [41]:
print("\nACTUAL PRODUCTION TARGETS TABLE COLUMNS")
print("=" * 70)

cursor.execute("DESCRIBE production_targets")

for row in cursor.fetchall():
    print(row)


ACTUAL PRODUCTION TARGETS TABLE COLUMNS
('target_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('machine_id', 'int', 'NO', 'MUL', None, '')
('target_date', 'date', 'NO', '', None, '')
('target_quantity', 'int', 'NO', '', None, '')


In [42]:
#  MACHINE-WISE TARGET ACHIEVEMENT
# ============================================================

print("MACHINE-WISE TARGET ACHIEVEMENT")
print("=" * 70)

machine_production = production.groupby("machine_id")["units_produced"].sum()
machine_target = targets.groupby("machine_id")["target_quantity"].sum()

machine_target_kpi = pd.DataFrame({
    "total_produced": machine_production,
    "total_target": machine_target
})

machine_target_kpi["target_achievement_percent"] = (
    machine_target_kpi["total_produced"] /
    machine_target_kpi["total_target"]
) * 100

machine_target_kpi["performance_status"] = machine_target_kpi[
    "target_achievement_percent"
].apply(
    lambda x: "Above Target" if x > 100
    else "On Target" if x >= 95
    else "Below Target"
)

machine_target_kpi["target_achievement_percent"] = (
    machine_target_kpi["target_achievement_percent"].round(2)
)

print(machine_target_kpi)

MACHINE-WISE TARGET ACHIEVEMENT
            total_produced  total_target  target_achievement_percent  \
machine_id                                                             
1                    84982         90426                       93.98   
2                    81096         80904                      100.24   
3                    71938         72129                       99.74   
4                    88769         90279                       98.33   
5                    83272         85606                       97.27   

           performance_status  
machine_id                     
1                Below Target  
2                Above Target  
3                   On Target  
4                   On Target  
5                   On Target  


In [43]:
# ============================================================
#  QUALITY KPIs


print("QUALITY KPIs")
print("=" * 70)

total_produced = production["units_produced"].sum()
total_rejected = production["units_rejected"].sum()

quality_defect_rate = (
    total_rejected / total_produced
) * 100

quality_pass_rate = (
    (total_produced - total_rejected) / total_produced
) * 100

print("\nTotal Units Produced:")
print(total_produced)

print("\nTotal Units Rejected:")
print(total_rejected)

print("\nDefect/Rejection Rate (%):")
print(round(quality_defect_rate, 2))

print("\nQuality Pass Rate (%):")
print(round(quality_pass_rate, 2))

QUALITY KPIs

Total Units Produced:
410057

Total Units Rejected:
18356

Defect/Rejection Rate (%):
4.48

Quality Pass Rate (%):
95.52


In [44]:
#  MACHINE-WISE DEFECT RATE
# ============================================================

print("MACHINE-WISE DEFECT RATE")
print("=" * 70)

machine_quality_kpi = production.groupby("machine_id").agg(
    total_produced=("units_produced", "sum"),
    total_rejected=("units_rejected", "sum")
)

machine_quality_kpi["defect_rate_percent"] = (
    machine_quality_kpi["total_rejected"] /
    machine_quality_kpi["total_produced"]
) * 100

machine_quality_kpi["quality_pass_rate_percent"] = (
    (machine_quality_kpi["total_produced"] -
     machine_quality_kpi["total_rejected"]) /
    machine_quality_kpi["total_produced"]
) * 100

machine_quality_kpi["defect_rate_percent"] = (
    machine_quality_kpi["defect_rate_percent"].round(2)
)

machine_quality_kpi["quality_pass_rate_percent"] = (
    machine_quality_kpi["quality_pass_rate_percent"].round(2)
)

print(machine_quality_kpi)

MACHINE-WISE DEFECT RATE
            total_produced  total_rejected  defect_rate_percent  \
machine_id                                                        
1                    84982            3848                 4.53   
2                    81096            3751                 4.63   
3                    71938            3213                 4.47   
4                    88769            4185                 4.71   
5                    83272            3359                 4.03   

            quality_pass_rate_percent  
machine_id                             
1                               95.47  
2                               95.37  
3                               95.53  
4                               95.29  
5                               95.97  


In [45]:

#  CHECK DOWNTIME DATA
# ============================================================

print("DOWNTIME COLUMNS")
print("=" * 70)

print(downtime.columns.tolist())

DOWNTIME COLUMNS
['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']


In [46]:
#  DOWNTIME KPIs
# ============================================================

print("DOWNTIME KPIs")
print("=" * 70)

total_downtime_hours = downtime["downtime_hours"].sum()

average_downtime_hours = downtime["downtime_hours"].mean()

total_downtime_events = downtime["downtime_id"].count()

print("\nTotal Downtime (Hours):")
print(round(total_downtime_hours, 2))

print("\nAverage Downtime per Event (Hours):")
print(round(average_downtime_hours, 2))

print("\nTotal Downtime Events:")
print(total_downtime_events)

DOWNTIME KPIs

Total Downtime (Hours):
70.86

Average Downtime per Event (Hours):
2.21

Total Downtime Events:
32


In [47]:

# MACHINE-WISE DOWNTIME
# ============================================================

print("MACHINE-WISE DOWNTIME")
print("=" * 70)

machine_downtime_kpi = downtime.groupby("machine_id").agg(
    total_downtime_hours=("downtime_hours", "sum"),
    downtime_events=("downtime_id", "count")
)

machine_downtime_kpi["average_downtime_hours"] = (
    machine_downtime_kpi["total_downtime_hours"] /
    machine_downtime_kpi["downtime_events"]
)

machine_downtime_kpi = machine_downtime_kpi.round(2)

print(machine_downtime_kpi)

MACHINE-WISE DOWNTIME
            total_downtime_hours  downtime_events  average_downtime_hours
machine_id                                                               
1                           7.06                4                    1.76
2                          15.49                6                    2.58
3                          26.39               13                    2.03
4                          16.30                6                    2.72
5                           5.62                3                    1.87


In [48]:

# KPI 5A: CHECK PRODUCTION TIME DATA
# ============================================================

print("PRODUCTION TIME SUMMARY")
print("=" * 70)

print("\nProduction Time Statistics:")
print(production["production_time_hours"].describe())

print("\nTotal Production Time (Hours):")
print(round(production["production_time_hours"].sum(), 2))

PRODUCTION TIME SUMMARY

Production Time Statistics:
count    450.0
mean       8.0
std        0.0
min        8.0
25%        8.0
50%        8.0
75%        8.0
max        8.0
Name: production_time_hours, dtype: float64

Total Production Time (Hours):
3600.0


In [49]:
# ============================================================
# MACHINE UTILIZATION
# ============================================================

print("MACHINE UTILIZATION KPIs")
print("=" * 70)

# Total available production hours for each machine
machine_available_hours = production.groupby("machine_id")[
    "production_time_hours"
].sum()

# Total downtime hours for each machine
machine_downtime_hours = downtime.groupby("machine_id")[
    "downtime_hours"
].sum()

# Combine available hours and downtime
machine_utilization_kpi = pd.DataFrame({
    "available_hours": machine_available_hours,
    "downtime_hours": machine_downtime_hours
}).fillna(0)

# Calculate actual operating hours
machine_utilization_kpi["operating_hours"] = (
    machine_utilization_kpi["available_hours"] -
    machine_utilization_kpi["downtime_hours"]
)

# Calculate utilization percentage
machine_utilization_kpi["utilization_percent"] = (
    machine_utilization_kpi["operating_hours"] /
    machine_utilization_kpi["available_hours"]
) * 100

machine_utilization_kpi["utilization_percent"] = (
    machine_utilization_kpi["utilization_percent"].round(2)
)

print(machine_utilization_kpi)

MACHINE UTILIZATION KPIs
            available_hours  downtime_hours  operating_hours  \
machine_id                                                     
1                     720.0            7.06           712.94   
2                     720.0           15.49           704.51   
3                     720.0           26.39           693.61   
4                     720.0           16.30           703.70   
5                     720.0            5.62           714.38   

            utilization_percent  
machine_id                       
1                         99.02  
2                         97.85  
3                         96.33  
4                         97.74  
5                         99.22  
